In [3]:
import os
from dotenv import load_dotenv
load_dotenv()

from openai import OpenAI

openai_client = OpenAI()
openai_client = OpenAI(
    api_key=os.getenv("OPENAI_API_KEY"),
    base_url=os.getenv("OPENAI_BASE_URL"),
)

In [2]:
from sqlitesearch import TextSearchIndex

sqlite_index = TextSearchIndex(
    text_fields=["question", "section", "answer"],
    keyword_fields=["course"],
    db_path="faq.db"
)

print(f"Loaded {sqlite_index.count()} documents from faq.db")

def search(query):
    boost_dict = {"question": 3.0, "section": 0.5}
    filter_dict = {"course": "llm-zoomcamp"}

    return sqlite_index.search(
        query,
        num_results=5,
        boost_dict=boost_dict,
        filter_dict=filter_dict
    )

# define the tool schema corresponding to the search function
search_tool = {
    "type": "function",
    "function": {
        "name": "search",
        "description": "Search the FAQ database for entries matching the given query.",
        "parameters": {
            "type": "object",
            "properties": {
                "query": {
                    "type": "string",
                    "description": "The search query to find relevant FAQ entries."
                }
            },
            "required": ["query"],
            "additionalProperties": False
        }
    }
}

Loaded 153 documents from faq.db


In [12]:
import json

def make_tool_call(tool_call):

    function_args = json.loads(tool_call.function.arguments)

    if tool_call.function.name == "search":
        search_results = search(**function_args)
    else:
        raise ValueError(f"Unknown tool: {tool_call.function.name}")

    return {
        "role": "tool",
        "call_id": tool_call.id,
        "content": json.dumps(search_results)
    }

In [5]:
res_test = openai_client.chat.completions.create(
    model=os.getenv("OPENAI_MODEL"),
    messages=[
        {"role": "user", "content": "Hi!"},
    ]
)

res_test

ChatCompletion(id='chatcmpl-ee01037b-9ec1-4010-93c1-d5feda467bf5', choices=[Choice(finish_reason='stop', index=0, logprobs=None, message=ChatCompletionMessage(content="\n\nHi there! How's it going? Is there anything I can help you with today?", refusal=None, role='assistant', annotations=None, audio=None, function_call=None, tool_calls=None, reasoning='Thinking Process:\n\n1.  **Analyze the Input:**\n    *   Input: "Hi!"\n    *   Intent: Greeting.\n    *   Tone: Friendly, casual.\n    *   Context: Starting a conversation.\n\n2.  **Determine the Appropriate Response:**\n    *   Acknowledge the greeting.\n    *   Offer assistance.\n    *   Keep it friendly and open-ended.\n    *   Match the tone (casual, helpful).\n\n3.  **Draft Potential Responses:**\n    *   Option 1: Hello! How can I help you today?\n    *   Option 2: Hi there! What\'s on your mind?\n    *   Option 3: Hey! How\'s it going? Anything I can do for you?\n    *   Option 4: Hello! Welcome. What would you like to chat about?

In [ ]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches.

Try to expand your search by using new keywords
based on the results you get from the search.

At the end, ask if there are other areas that the user wants to explore.
""".strip()


def agent_loop(instructions, question) -> str:

    messages = [
        {"role": "developer", "content": instructions},
        {"role": "user", "content": question},
    ]

    it = 1
    while True:
        print(f"\n\nIteration {it}:\n")


        res = openai_client.chat.completions.create(
            model=os.getenv("OPENAI_MODEL"),
            messages=messages,
            tools=[search_tool]
        )

        messages.append(res.choices[0].message)
        has_tool_calls = False

        for item in res.choices:
            if item.finish_reason == "tool_calls":
                for tool_call in item.message.tool_calls:
                    print(f"Tool call: {tool_call.function.name} with arguments: {tool_call.function.arguments}")
                    messages.append(make_tool_call(tool_call))
                    has_tool_calls = True

            elif item.finish_reason == "stop":
                print(f"Message: {item.message.content}")
                last_message = item.message.content
                messages.append(item.message)

        it += 1
        if has_tool_calls is False:
            break

    return last_message


# agent_loop(instructions, "How do I run Olama locally?")
agent_loop(instructions, "I just discovered the course. Can I join it?")




Iteration 1:

Tool call: search with arguments: {"query": "Olama locally"}
Tool call: search with arguments: {"query": "Ollama local installation"}
Tool call: search with arguments: {"query": "run Ollama"}


Iteration 2:

Message: 

Based on my search results, I found information about **Ollama** (which I believe is what you meant by "Olama"). Here's how to run Ollama locally:

## Installing Ollama

### Download and Install
Visit [https://ollama.com/download](https://ollama.com/download) and choose your operating system:

- **macOS**: Download the `.pkg` and install it
- **Windows**: Download the `.msi` and install it
- **Linux**: Run the following command in the terminal:
  ```bash
  curl -fsSL https://ollama.com/install.sh | sh
  ```

### Running Ollama Locally

Once installed, open a terminal and run:
```bash
ollama run llama3
```

This command will:
- Download the LLaMA 3 model (~4GB)
- Start the model locally
- Open a chat-like interface where you can type questions

### Testing

"\n\nGreat news! **Yes, you can still join the course.**\n\nHere's what you need to know:\n\n## Joining the Course\n- You don't need to register to start learning - you can begin working through the material and submitting homework right away\n- Registration is mainly used to gauge interest before the course start date\n- You're already accepted to the course\n\n## Getting a Certificate\nIf you want to receive a certificate, there are some important requirements:\n- You need to submit your capstone project while the course is still accepting submissions\n- You must complete the required peer reviews\n- You can work through the material at your own pace, but project submission and peer review must happen during a live cohort period\n- Homework is not mandatory for the certificate, but it does count toward your leaderboard rank\n\n## Important Note\nThe next course offering will be in **Summer 2027**, so if you're looking to join now, make sure to check when the current cohort's project 

In [22]:
instructions = """
You're a course teaching assistant.
You're given a question from a course student and your task is to answer it.

If you want to look up information, use the search function. 
Use as many keywords from the user question as possible when making first requests.

Make multiple searches. First perform search, analyze the results 
and then perform more searches. 

The question has to be about the course or its logistics, offtopic questions 
shouldn't be answered. If the search returns nothing, it's likely an off-topic question.
If you can't answer the question using FAQ, don't do it yourself. Only use the 
facts from the FAQ database.

At the end, ask if there are other areas that the user wants to explore.
""".strip()

agent_loop(instructions, "what's queen gambit?")



Iteration 1:

Tool call: search with arguments: {"query": "queen gambit"}


Iteration 2:

Tool call: search with arguments: {"query": "queen"}


Iteration 3:

Tool call: search with arguments: {"query": "gambit"}


Iteration 4:

Message: 

I searched the FAQ database for information about "queen gambit" but found no relevant results. This appears to be an off-topic question that isn't related to the course content or logistics.

The "Queen's Gambit" is typically known as either:
- A chess opening
- A popular Netflix TV series

Neither of these topics appears to be covered in this course based on the available FAQ information.

Do you have any questions about the course material, assignments, deadlines, or other course-related topics that I can help you with?


'\n\nI searched the FAQ database for information about "queen gambit" but found no relevant results. This appears to be an off-topic question that isn\'t related to the course content or logistics.\n\nThe "Queen\'s Gambit" is typically known as either:\n- A chess opening\n- A popular Netflix TV series\n\nNeither of these topics appears to be covered in this course based on the available FAQ information.\n\nDo you have any questions about the course material, assignments, deadlines, or other course-related topics that I can help you with?'